# General Normal Forward Example

Define a symbolic general problem with `ProblemBuilder`, fix inverse-style coefficients to known values, generate labels, then train in forward mode through `builder.run(...)`.


## Setup
Import the symbolic builder and create helpers for notebook-local run folders and summaries.


In [1]:
from argparse import Namespace
from pathlib import Path
import json

from kkthn.builder import ProblemBuilder


ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent


def make_output_root(name):
    output_root = ROOT / "notebooks" / "_runs" / name
    output_root.mkdir(parents=True, exist_ok=True)
    return output_root


def latest_run_dir(output_root):
    runs = [path for path in output_root.iterdir() if path.is_dir()]
    if not runs:
        raise FileNotFoundError(f"No run directories found under {output_root}")
    return max(runs, key=lambda path: path.stat().st_mtime)


def load_summary(run_dir):
    with open(run_dir / "summary.json", "r", encoding="utf-8") as fh:
        return json.load(fh)


def builder_args(mode, data, config, output_root):
    return Namespace(
        action="run",
        mode=mode,
        samples=data.get("num_samples"),
        epochs=config.get("epochs"),
        batch_size=config.get("batch_size"),
        seed=data.get("seed"),
        noise_scale=data.get("noise_scale"),
        output_dir=str(output_root),
    )


## Configuration
Edit these dictionaries to change the parameter box, fixed coefficients, training loop, or projection settings.


In [2]:
DATA = {
    "type": "general_normal",
    "mode": "forward",
    "num_samples": 12,
    "seed": 42,
    "noise_scale": 0.0,
    "x_L": [-1.0, -1.0],
    "x_U": [1.0, 1.0],
    "inv_param": ["a0", "a1"],
    "inv_param_label": [1.0, -1.0],
}

CONFIG = {
    "epochs": 3,
    "batch_size": 4,
    "learning_rate": 1e-3,
    "train_frac": 0.8,
    "hidden_size": 32,
    "hidden_layers": 2,
    "seed": 42,
    "dtype": "float64",
    "print_every": 1,
}

PROJ = {
    "fb_eps": 1e-8,
    "gn_max_iters": 25,
    "gn_tol": 1e-8,
}


## Problem Definition
The builder keeps the algebraic constraints readable while compiling them into the internal JAX model.


In [3]:
def build_normal_problem(data):
    builder = ProblemBuilder(y_bound=4.0)
    x = builder.add_parameter(["x1", "x2"])
    theta = builder.add_inverse_parameter(data["inv_param"])
    y = builder.add_variable(["y1", "y2", "y3"])

    builder.objective = 0.5 * (y.y1**2 + y.y2**2 + y.y3**2)
    builder.constraints.add(
        theta.a0 * y.y1 + y.y2 - x.x1 == 0,
        y.y2 - theta.a1 * y.y3 - x.x2 == 0,
        y.y1**2 + y.y3**2 <= 2.0,
    )
    builder.bounds.set(lower=-4.0, upper=4.0)
    return builder


## Train
Create the builder instance and call `builder.run(...)`; the runner handles label generation, artifacts, and training.


In [4]:
output_root = make_output_root("general_normal_forward")
args = builder_args("forward", DATA, CONFIG, output_root)
builder = build_normal_problem(DATA)
exit_code = builder.run(args, root=ROOT, data=DATA, train=CONFIG, proj=PROJ)
run_dir = latest_run_dir(output_root)
summary_json = load_summary(run_dir)

print(f"Exit code: {exit_code}")
print(f"Run directory: {run_dir}")
print(f"Saved artifacts: {summary_json['artifacts']}")


KKT-HardNet
  dims: n_x=2 n_y=3 n_eq=2 n_ineq=7
  mode: forward
  samples: train=9 val=3 batch_size=4
  network: [2, 32, 32, 3]
epoch 0001 | train=2.143790e-02 val=1.752569e-02 raw_val=3.614144e-01 eq=7.080e-12 ineq=0.000e+00 | train_batch=1.0367s val_batch=0.0011s
epoch 0002 | train=1.536444e-02 val=1.401611e-02 raw_val=3.633189e-01 eq=7.168e-12 ineq=0.000e+00 | train_batch=0.0012s val_batch=0.0015s
epoch 0003 | train=1.009720e-02 val=1.084991e-02 raw_val=3.643570e-01 eq=7.197e-12 ineq=0.000e+00 | train_batch=0.0013s val_batch=0.0018s
Saved run artifacts: D:\Projects\KKTHardNet\notebooks\_runs\general_normal_forward\builder_general_forward_20260414_121412
Training time: 6.258s
=== Timing summary ===
Train step time total      : 3.113s
Validation time total      : 0.004s
Avg train time / epoch     : 1.0375s
Avg validation time / epoch: 0.0015s
Avg train time / batch     : 0.3458s
Avg validation time / batch: 0.0015s
=== Profiled component time distribution ===
Backbone forward :   0.06

## Summary
Read the emitted `summary.json` from the newest run folder.


In [5]:
summary = {
    "output_dir": str(run_dir),
    "dims": summary_json["dims"],
    "final": summary_json["final"],
    "component_time_percent": summary_json["timing_profile"]["component_time_percent"],
}
print(json.dumps(summary, indent=2))


{
  "output_dir": "D:\\Projects\\KKTHardNet\\notebooks\\_runs\\general_normal_forward\\builder_general_forward_20260414_121412",
  "dims": {
    "n_eq": 2,
    "n_ineq": 7,
    "n_x": 2,
    "n_y": 3,
    "n_z": 19,
    "raw_n_ineq": 1
  },
  "final": {
    "epoch": 3,
    "mean_batch_loss": 0.01103869204083606,
    "train_batches": 3,
    "train_epoch_time_sec": 0.003979000000001065,
    "train_eq_l2": 1.1820543106133397e-11,
    "train_eval_time_sec": 0.001030100000001255,
    "train_ineq_l2": 0.0,
    "train_loss": 0.01009720110658318,
    "train_raw_mse": 0.7544435910933908,
    "train_step_time_per_batch_sec": 0.0007329333333316868,
    "train_step_time_sec": 0.0021987999999950603,
    "train_time_per_batch_sec": 0.0013263333333336884,
    "val_eq_l2": 7.197461991331102e-12,
    "val_ineq_l2": 0.0,
    "val_loss": 0.010849913509478074,
    "val_raw_mse": 0.3643570150514746,
    "validation_batches": 1,
    "validation_epoch_time_sec": 0.0018232000000004689,
    "validation_time_pe